# Python-Compatible Interface Design for Standardized MATLAB Outputs

## Overview

The standardized **stimuli.mat** dataset created in the previous stage preserved the full analytical structure of each recording session, but the resulting MATLAB format was not yet ideal for direct use in the Python-based analysis workflow developed later in the project. A separate interface layer was therefore required to reorganize key fields into a simpler Python-compatible structure without changing the analytical meaning of the dataset.

This stage focused on building that interface. Rather than limiting the workflow to a single hard-coded file, the conversion tool was designed to allow user-selected MATLAB inputs so that the same restructuring logic could be applied across different reconstructed session files. The result was a lightweight and reusable bridge from MATLAB-based reconstruction to Python-based validation, comparison, and downstream analysis.

## Why a Python-Compatible Interface Was Required

Although **stimuli.mat** served as a standardized dataset after reconstruction, the internal structure still reflected MATLAB-oriented storage logic. In particular, version 7.3 `.mat` files required HDF5-based loading and often introduced unnecessary complexity when only a subset of fields was needed for Python analysis. Nested structures and MATLAB-specific formatting increased the overhead of direct access in Python and made repeated downstream use less efficient.

The problem at this stage was therefore not the quality of the reconstructed dataset itself, but the interface through which the dataset could be reused. A Python-compatible layer was needed to preserve the essential fields, flatten the structure, and make repeated access easier across multiple user-selected session files.

## Interface Restructuring Strategy

The purpose of the interface was not to redefine the dataset, but to reorganize it for more efficient downstream access. Core signal arrays, event timestamps, and sampling information were extracted from the original MATLAB structure and saved into a simplified flat schema that could be loaded directly in Python with minimal parsing overhead.

To support repeated use across sessions, the conversion step was implemented as a user-selectable file transformation workflow rather than as a one-time file export. That design made the interface portable across reconstructed MATLAB outputs while keeping the downstream Python environment consistent and easy to use.

In [8]:
# Open a file picker for selecting a reconstructed MATLAB dataset
# Build the output path for the Python-compatible export
# Load the MATLAB v7.3 structure
# Extract fields required for downstream Python analysis
# Save the flattened dataset as a new .mat file

import numpy as np
import hdf5storage
import tkinter as tk
import os
from tkinter import filedialog
from scipy.io import savemat

root = tk.Tk()
root.withdraw()
root.attributes('-topmost', True)

# Select the standardized session-level MATLAB dataset generated in the previous stage
input_path = filedialog.askopenfilename(
    title="Select stimuli.mat file",
    filetypes=[("MATLAB files", "*.mat")]
)
if not input_path:
    raise Exception("No input file selected!")
print("Input:", input_path)

input_dir = os.path.dirname(input_path)
input_name = os.path.basename(input_path)
base_name, _ = os.path.splitext(input_name)
output_path = os.path.join(input_dir, base_name + "_python.mat")

print("Auto-saving as:", output_path)

# Load the MATLAB v7.3 dataset using an HDF5-compatible reader
data = hdf5storage.loadmat(input_path)
stim = data['stimuli']

# Extract the signal arrays, event timestamps, and metadata required for downstream Python-based analysis
photo1 = stim['photo1'][0, 0].flatten()
photo2 = stim['photo2'][0, 0].flatten()
cue1times = stim['cue1times'][0, 0].flatten()
sol1times = stim['sol1times'][0, 0].flatten()
downsamplingrate = stim['downsamplingrate'][0, 0].flatten()[0]

# Flatten nested MATLAB fields into directly accessible Python arrays
stimuli_python = {
    "photo1": photo1.astype(np.float32),
    "photo2": photo2.astype(np.float32),
    "cue1times": cue1times.astype(np.float32),
    "sol1times": sol1times.astype(np.float32),
    "downsamplingrate": np.array([[int(downsamplingrate)]], dtype=np.int32)
}

# Save a lightweight MATLAB file that can be loaded directly through scipy.io.loadmat() in downstream Python scripts
savemat(output_path, stimuli_python)

print("Conversion complete!")
print("Saved:", output_path)


Input: C:/Users/admin/python_project/stimuli_file/gcajr/Flna30_473+561nm_stimuli.mat
Auto-saving as: C:/Users/admin/python_project/stimuli_file/gcajr\Flna30_473+561nm_stimuli_python.mat
Conversion complete!
Saved: C:/Users/admin/python_project/stimuli_file/gcajr\Flna30_473+561nm_stimuli_python.mat


## Compatibility Validation

After conversion, the restructured dataset was loaded again through the standard `scipy.io.loadmat()` workflow. The major arrays could now be accessed directly without HDF5-specific handling or repeated navigation through nested MATLAB fields.

This validation step confirmed that the converted structure was not only lighter in form, but immediately usable in a standard Python environment. In practical terms, the restructuring established a Python-compatible analysis interface while preserving the standardized content of the original MATLAB dataset.

In [ ]:
# Open a file picker for selecting the converted Python-compatible dataset
# Load the converted file with scipy.io.loadmat
# Read core signals and event timestamps for compatibility checking
# Print basic structural information

import numpy as np
import tkinter as tk
from tkinter import filedialog
from scipy.io import loadmat

root = tk.Tk()
root.withdraw()
root.attributes('-topmost', True)

mat_path = filedialog.askopenfilename(
    title="Select stimuli_python.mat file",
    filetypes=[("MATLAB files", "*.mat")]
)

if not mat_path:
    raise Exception("No file selected!")

print("Loaded:", mat_path)

data = loadmat(mat_path)

photo1 = data['photo1'].flatten()
photo2 = data['photo2'].flatten()
sol1times = data['sol1times'].flatten()
cue1times = data['cue1times'].flatten()
downsamplingrate = int(data['downsamplingrate'][0][0])

print("Validation load completed.")

## Structure Verification Through Event-Aligned Check

A reward-aligned plotting step was used to verify that the converted dataset preserved the temporal structure required for downstream analysis. Signal segments were extracted relative to reward delivery, baseline-adjusted, and averaged across trials to confirm that the event-aligned organization remained intact after MATLAB-to-Python restructuring.

The purpose of this step was not final biological interpretation, but structural verification. Specifically, it confirmed that the converted interface preserved signal continuity, event timing relationships, and analysis compatibility after transfer into the Python environment.

In [ ]:
# Define the reward-aligned analysis window
# Extract signal segments around each reward timestamp
# Apply baseline normalization within each segment
# Average segments across trials for structural validation
# Plot the converted signals to confirm alignment integrity

import numpy as np
import matplotlib.pyplot as plt

# Define the reward-aligned validation window used for structure verification
pre, post = 2, 6

n_trials = len(sol1times)

signal1_segments = []
signal2_segments = []

# Extract reward-aligned signal segments to verify that event timing and signal continuity were preserved after interface conversion
for t in sol1times:
    idx = int(t * downsamplingrate)

    start = idx - int(pre * downsamplingrate)
    end = idx + int(post * downsamplingrate)

    if start >= 0 and end < len(photo1):
        # Use the -3 to -2 s interval as the baseline window for segment normalization
        base_start = idx - int(3 * downsamplingrate)
        base_end = idx - int(2 * downsamplingrate)

        baseline1 = np.mean(photo1[base_start:base_end])
        baseline2 = np.mean(photo2[base_start:base_end])

        segment1 = photo1[start:end] - baseline1
        segment2 = photo2[start:end] - baseline2

        signal1_segments.append(segment1)
        signal2_segments.append(segment2)

signal1_segments = np.array(signal1_segments)
signal2_segments = np.array(signal2_segments)

time_vec = np.linspace(-pre, post, signal1_segments.shape[1])

# Plot trial-averaged reward-aligned traces as a structural validation check
plt.plot(time_vec, signal1_segments.mean(axis=0), label='Signal 1')
plt.plot(time_vec, signal2_segments.mean(axis=0), label='Signal 2')
plt.axvline(0, linestyle='--')
plt.title('Reward-aligned validation plot')
plt.xlabel('Time (s)')
plt.ylabel('Baseline-adjusted intensity')
plt.legend()
plt.grid()
plt.tight_layout()
plt.show()

## Outcome

This stage established a reusable interface layer between MATLAB-based reconstruction and Python-based analysis. By flattening the standardized MATLAB dataset into a simpler schema and allowing user-selected input files to be converted with the same logic, the workflow reduced environment-specific loading complexity without changing the meaning of the original session data.

The converted output made later Python stages easier to execute and compare across sessions, and it directly supported the next phase of the project: dual-color signal inspection, interference analysis, and algorithm-oriented evaluation in a unified Python workflow.